# Prepare CHP data

This prepares yearly CHP generation in MWh per country based on the Eurostat CHP questionaire from https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/

- per technology shares are calculated using OPSD data where available

- For other countries, we create shares based on yearly entsoe generation data from parse_generation_entsoe_sftp

- Furthermore, we prepare yearly CHP profile from original data (lion's?)

Careful: '../parsed_data/generation_'+year+'_annual_entsoe.csv' is needed for 2015-2017!

In [1]:
import pandas as pd
import numpy as np
import wget
import os

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
download = "no"

In [3]:
dir_in = "../source_data/chp/"
dir_out = "../parsed_data/"
years = ['2015','2016','2017'] #careful if extending list of years as SK 2014 is missing
fn = "CHPdata2005-2017.xlsx"
fn_heat_dem = "../source_data/chp/chp_data.xlsx"
fn_opsd = "conventional_power_plants_EU.csv"
url = "https://ec.europa.eu/eurostat/documents/38154/4956229/CHPdata2005-2017.xlsx/871cc151-5733-423f-ae38-de9b733aa81e"
url_opsd = 'https://data.open-power-system-data.org/conventional_power_plants/2020-10-01/conventional_power_plants_EU.csv'

In [4]:
#only if data has to be downloaded again:
if download == "yes":
    os.remove(dir_in+fn)
    wget.download(url,dir_in+fn)

! if data is downloaded again, sheet 2017 needs "2017" in cell A3, otherwise, 2017 values are missing

! should also copy all PJ data to row 37 to be consistent

In [5]:
map_country_ISO = {
	"Austria" : "AT",
	"Belgium" : "BE",
	"Belgium1" : "BE",
	"Bulgaria" : "BG",
	"Croatia" : "HR",
	"Cyprus" : "CY",
	"Czech" : "CZ",
	"Czech Republic" : "CZ",
	"Czechia" : "CZ",
	"Denmark" : "DK",
	"Estonia" : "EE",
	"Estonia2" : "EE",
	"Finland" : "FI",
	"France" : "FR",
	"Germany" : "DE",
	"Germany1" : "DE",
	"Germany1, 2" : "DE",
	"Greece" : "GR",
	"Greece2" : "GR",
	"Hungary" : "HU",
	"Hungary1" : "HU",
	"Hungary3" : "HU",
	"Ireland" : "IE",
	"Ireland1" : "IE",
	"Italy" : "IT",
	"Latvia" : "LV",
	"Lithuania" : "LT",
	"Luxembourg" : "LU",
	"Malta" : "MT",
	"Netherlands" : "NL",
	"Norway" : "NO",
	"Norway2" : "NO",
	"Poland" : "PL",
	"Portugal" : "PT",
	"Portugal1" : "PT",
	"Romania" : "RO",
	"Slovakia" : "SK",
	"Slovakia3" : "SK",
	"Slovenia" : "SI",
	"Slovenia1" : "SI",
	"Spain" : "ES",
	"Sweden" : "SE",
	"Sweden2" : "SE",
	"United Kingdom" : "GB",
    "Estonia*" : "EE"
}

In [6]:
dict_opsd_tech = {
    'Natural gas':'Gas',
    'Oil':'Oil',
    'Biomass and biogas':'Biomass',
    'Nuclear':'Nuclear',
    'Non-renewable waste':'Other',
    'Mixed fossil fuels':'Other',
    'Hard coal':'HardCoal',
    'Other or unspecified energy sources':'Other',
    'Lignite':'Lignite',
    'Bioenergy':'Biomass',
    'Other fossil fuels':'Other',
    'Other fuels':'Other',
    'Waste':'Other'
}

In [7]:
#opsd only has chp data for the following countries
opsd_countries = ['BE', 'FI', 'ES', 'SE', 'SI', 'AT', 'DE']

In [8]:
#lame fix for column naming:
year_to_country = {
    2015 : "country",
    2016 : "country",
    2017 : "country",
}

In [10]:
df=pd.DataFrame()
for year in years:
    df_temp = pd.read_excel(dir_in + fn, sheet_name = year,
                            header=2, usecols = "A:E", nrows=30,na_values=":") 
    df_temp = df_temp.rename(columns = year_to_country)
    df_temp['year'] = year
    df = pd.concat([df, df_temp], ignore_index=True)

Rename countries to ISO and remove non listed countries or regions

In [11]:
df.index = df['country']
df = df.rename(index=map_country_ISO)
df = df[df['country'].isin(map_country_ISO)].drop(columns='country').reset_index()

Convert CHP generation to MWh and drop non needed columns

In [12]:
df['CHP_MWh'] = df['CHP electricity generation, TWh']*1000*1000
df_chp_gen = df[['year','country','CHP_MWh']].copy().set_index(['year','country'])

now we copy 2016 data for NO as this is missing in 2017

In [13]:
df_chp_gen.loc[('2017','NO'),:] = df_chp_gen.loc[('2016','NO'),:]

In [14]:
df_chp_gen.head(1)

,,CHP_MWh
year,country,
2015,BE,12479000.0


Now we load OPSD data to calculate shares per technology

we calculate shares per technology based on their share in yearly generation

In [26]:
chp_techs = ['Biomass','Gas','HardCoal','Lignite','Nuclear','Oil','Other']

In [27]:
df_gen = pd.DataFrame()
for year in years:
    df_gen_temp = pd.read_csv('../parsed_data/generation_'+year+'_annual_entsoe.csv')
    df_gen_temp['year'] = year
    df_gen = pd.concat([df_gen, df_gen_temp], ignore_index=True)
df_gen = df_gen[df_gen.tech.isin(chp_techs)]
df_gen = df_gen.groupby(['year','country','tech']).sum()[['net_generation']]
df_gen.head()

net_generation
year country tech                    
2015 AT      Biomass         2.411676
             Gas             7.641010
             HardCoal        1.675485
             Oil             0.000000
             Other           1.994389

In [28]:
df_gen_total = df_gen.reset_index().groupby(['year','country']).sum().drop(columns='tech')
df_gen_total.head()

net_generation
year country                
2015 AT            13.722561
     BE            52.575210
     BG            37.133988
     CH            17.668990
     CY             1.666379

In [29]:
df_gen_shares = df_gen.reset_index().merge(df_gen_total.reset_index(), how='left', on=['year','country'])
df_gen_shares['chp_share'] = df_gen_shares['net_generation_x'] / df_gen_shares['net_generation_y']
df_gen_shares = df_gen_shares[['year','country','tech','chp_share']]
df_gen_shares = df_gen_shares.pivot_table(index=['year','country'],columns='tech',values='chp_share') 
df_gen_shares.head()

tech           Biomass       Gas  HardCoal   Lignite   Nuclear       Oil  \
year country                                                               
2015 AT       0.175745  0.556821  0.122097       NaN       NaN  0.000000   
     BE       0.042130  0.364386  0.035298  0.000000  0.434318  0.000067   
     BG       0.004642       NaN       NaN  0.603388  0.391970       NaN   
     CH            NaN  0.001405       NaN       NaN  0.998595       NaN   
     CY            NaN       NaN       NaN       NaN       NaN  1.000000   

tech             Other  
year country            
2015 AT       0.145337  
     BE       0.123801  
     BG            NaN  
     CH            NaN  
     CY            NaN

now merge this into chp df, calculate the yearly CHP generation per technology and clean up

In [30]:
df_chp_tech_merged = df_chp_gen.merge(df_gen_shares, how='left',on=['year','country'])
df_chp_tech_merged.head()

CHP_MWh   Biomass       Gas  HardCoal   Lignite   Nuclear  \
year country                                                                 
2015 BE       12479000.0  0.042130  0.364386  0.035298  0.000000  0.434318   
     BG        2945000.0  0.004642       NaN       NaN  0.603388  0.391970   
     CZ       11785000.0  0.031892  0.023299  0.067678  0.449456  0.356449   
     DK       11568000.0  0.028039  0.286610  0.557477       NaN       NaN   
     DE       78787000.0  0.087831  0.036993  0.197529  0.333277  0.213724   

                   Oil     Other  
year country                      
2015 BE       0.000067  0.123801  
     BG            NaN       NaN  
     CZ       0.000711  0.070516  
     DK       0.015778  0.112096  
     DE       0.004323  0.126324

In [31]:
df_chp_tech_merged['HardCoal'] = df_chp_tech_merged['HardCoal']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Biomass'] = df_chp_tech_merged['Biomass']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Other'] = df_chp_tech_merged['Other']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Gas'] = df_chp_tech_merged['Gas']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Lignite'] = df_chp_tech_merged['Lignite']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged['Oil'] = df_chp_tech_merged['Oil']  * df_chp_tech_merged['CHP_MWh']
df_chp_tech_merged = df_chp_tech_merged.drop(columns='CHP_MWh')
df_chp_tech_merged.head()

Biomass           Gas      HardCoal       Lignite  \
year country                                                           
2015 BE       5.257388e+05  4.547173e+06  4.404847e+05  0.000000e+00   
     BG       1.367133e+04           NaN           NaN  1.776978e+06   
     CZ       3.758420e+05  2.745805e+05  7.975811e+05  5.296841e+06   
     DK       3.243608e+05  3.315504e+06  6.448894e+06           NaN   
     DE       6.919902e+06  2.914561e+06  1.556268e+07  2.625789e+07   

               Nuclear            Oil         Other  
year country                                         
2015 BE       0.434318     835.497463  1.544918e+06  
     BG       0.391970            NaN           NaN  
     CZ       0.356449    8376.524041  8.310276e+05  
     DK            NaN  182517.255529  1.296724e+06  
     DE       0.213724  340622.334363  9.952680e+06

In [32]:
df_chp_tech_long = df_chp_tech_merged.reset_index().melt(id_vars=['year','country'],
                                           var_name="tech",
                                           value_name="MWh")
df_chp_tech_long.head()

,year,country,tech,MWh
0,2015,BE,Biomass,5.257388e+05
1,2015,BG,Biomass,1.367133e+04
2,2015,CZ,Biomass,3.758420e+05
3,2015,DK,Biomass,3.243608e+05
4,2015,DE,Biomass,6.919902e+06


## Now we also create hourly profiles

In [33]:
df_heat_demand_in = pd.read_excel(fn_heat_dem)
df_heat_demand_in.head(1)

,date,heat_demand
0,2014-01-01 00:00:00+00:00,0.86235


In [34]:
df_heat_demand = df_heat_demand_in.drop("date", axis = 1)
df_heat_demand["heat_demand_relative"] = df_heat_demand["heat_demand"]/df_heat_demand["heat_demand"].sum()
df_heat_demand.tail()

,heat_demand,heat_demand_relative
8755,0.896596,0.000153
8756,1.000000,0.000171
8757,1.000000,0.000171
8758,1.000000,0.000171
8759,1.000000,0.000171


# Export both to csv

In [35]:
df_chp_tech_long.to_csv(dir_out + "chp_generation.csv", encoding="utf-8", index=False)

In [36]:
df_heat_demand.to_csv(dir_out + "heat_demand.csv", encoding="utf-8", index = False)